# Provenance Agent — Quickstart

This notebook walks through the core functions of the provenance agent. Given any Jupyter notebook, the agent identifies which Python libraries are imported and generates an APA bibliography for the software used.

**Functions covered:**
- `parse_notebook` — reads a `.ipynb` file and returns all imported libraries
- `extract_libraries` — extracts imports from a single string of Python code
- `strip_ipython_directives` — removes `%magic` and `!shell` lines before parsing
- `add_bibliography_to_notebook` — appends an APA bibliography as a markdown cell

## Setup

Add `src/` to the path so the module can be imported without installing the package.

In [3]:
import sys
sys.path.insert(0, '../src')

%pip install nbformat pybtex pyyaml -q

from notebook_parser import parse_notebook, extract_libraries, strip_ipython_directives

python(77058) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Note: you may need to restart the kernel to use updated packages.


## 1. `parse_notebook` — scan a full notebook

This is the main function you'll call. Pass a path to any `.ipynb` file and it returns the sorted list of top-level libraries imported anywhere in the notebook. Called with no argument inside a running Jupyter session, it auto-detects the current notebook path (requires `pip install ipynbname`).

In [ ]:
result = parse_notebook('start_here.ipynb')
print('Libraries found:', result['libraries'])

You can also point it at any other notebook by passing an explicit path:

```python
result = parse_notebook('sample.ipynb')
```

## 2. `extract_libraries` — parse a snippet of code

`extract_libraries` works on a raw string rather than a file. It's useful for testing or for processing individual cells. It handles both `import X` and `from X.Y import Z` forms, and always returns the top-level package name (e.g. `matplotlib.pyplot` → `matplotlib`).

In [ ]:
code = """
import numpy as np
import pandas as pd
from matplotlib.pyplot import plt
"""
print(extract_libraries(code))

## 3. `strip_ipython_directives` — clean up magic and shell commands

Paleoclimate notebooks commonly use `%matplotlib inline`, `!pip install ...`, and similar IPython-only syntax. These lines are not valid Python and would cause `ast.parse` to fail. `strip_ipython_directives` removes them so that import extraction can proceed on the rest of the cell.

In [ ]:
raw = """
%matplotlib inline
!pip install pyleoclim
import pyleoclim as pyleo
"""
print(strip_ipython_directives(raw))

The `%` and `!` lines are dropped; the `import` line passes through unchanged. `parse_notebook` calls this automatically on every cell before parsing.

## 4. Dataset Extraction _(coming soon)_

In addition to libraries, the provenance agent will eventually identify datasets loaded in a notebook. Data in these notebooks typically comes from three sources:

- **LiPDGraph** — a graph database queried directly
- **PyLiPD** — local or web-based LiPD files
- **PANGAEA / NOAA NCEI** — datasets fetched via PyleoTUPS

For now, `parse_notebook` returns an empty list as a placeholder.

In [ ]:
result = parse_notebook('start_here.ipynb')
print('Datasets:', result['datasets'])  # TODO: implement dataset extraction

## 5. `add_bibliography_to_notebook` — append APA citations to a notebook

`add_bibliography_to_notebook` takes a notebook path, extracts its libraries, generates an APA 7th edition bibliography via pandoc, and appends it as a new markdown cell at the end of the notebook file. It includes both paper and software citations where available, deduplicates by DOI, and flags libraries without citations.

In [ ]:
from bibliography import add_bibliography_to_notebook
import shutil

# Work on a copy so we don't modify the original
shutil.copy('sample.ipynb', '/tmp/sample_with_bib.ipynb')

add_bibliography_to_notebook('/tmp/sample_with_bib.ipynb')
print('Bibliography cell appended to /tmp/sample_with_bib.ipynb')

---
## Testing on Real Notebooks

The five notebooks below are from the *Comparing Simulated and Reconstructed Climate Variability over the Past Millennium* collection. Each one loads CMIP6/PMIP model output and LMR reconstruction data via `intake`, `xarray`, and `zarr`. Running `parse_notebook` on them exercises the parser against real scientific notebooks.

### C02_b — Data Assimilation with Individual Seasonality

This notebook uses `cfr` (Climate Field Reconstruction), `numpy`, and `pickle`. It exercises the parser on a paleoclimate-specific workflow that includes serialized data loading.

In [ ]:
result = parse_notebook('C02_b_DA_with_individual_seasonality.ipynb')
print(f'Libraries: {result["libraries"]}')
assert result['libraries'] == ['cfr', 'numpy', 'pickle'], f'Unexpected: {result["libraries"]}'
print('PASSED')

In [ ]:
notebooks = [
    'comparing-simulated-reconstructed-climate/CMIP6_LMR.ipynb',
    'comparing-simulated-reconstructed-climate/data_from_esm_cloudcat.ipynb',
    'comparing-simulated-reconstructed-climate/spatial_snapshots_xarray_bonuses.ipynb',
    'comparing-simulated-reconstructed-climate/VICS_dashboard.ipynb',
    'comparing-simulated-reconstructed-climate/widget_primer.ipynb',
]

for nb_path in notebooks:
    result = parse_notebook(nb_path)
    print(f'{nb_path.split("/")[-1]}')
    print(f'  Libraries: {result["libraries"]}')
    print()